# System 2: Single-Agent RAG (Showcase)

This notebook demonstrates **System 2 (Single-Agent RAG)**. 
Unlike System 1 (Monolith), which always follows a fixed `retrieve -> generate` pipeline, System 2 uses a **ReAct Agent** with access to specific tools.

The agent autonomously decides *when* to search, *how* to search, and can perform mathematical operations. Crucially, the **retrieval stack (chunking, vectorstore, hybrid index, reranker) is identical** to System 1 to ensure a fair architectural comparison.

## Available Tools:
1. `list_filings()`: Check which SEC 10-Ks are available in the knowledge base.
2. `search_section(ticker, fiscal_year, section, sub_query=None)`: Target a specific section (e.g., MD&A) using ChromaDB metadata filtering. The optional `sub_query` drives query-specific ranking within the filtered section (via FlashRank cross-encoder).
3. `retrieve_chunks(query)`: Fallback to general hybrid semantic search across all docs.
4. `calculate(expression)`: Safe math evaluator (`simpleeval`) to prevent LLM math hallucinations.

## Reflection Verifier

After the ReAct loop produces a draft answer, an external **Reflexion-style verifier** (Shinn et al. 2023, arXiv:2303.11366) evaluates the answer against four dimensions (groundedness, numerical accuracy, completeness, citations). On `status="revise"` the agent runs **one** more iteration with reviewer feedback injected as a `HumanMessage`. Reflection is enabled by default and can be disabled via `AgentRAGPipeline(config_override={"reflection_enabled": False})` for ablation runs.


In [ ]:
import os
import sys
from pathlib import Path

# Ensure src/ is in the python path
if Path("src").exists():
    sys.path.append(os.path.abspath("."))
else:
    sys.path.append(os.path.abspath("../../"))

from dotenv import load_dotenv

load_dotenv()

from src.common.ingestion import ProcessedFiling
from src.systems.rag_agent.pipeline import AgentRAGPipeline

import warnings
warnings.filterwarnings('ignore')


## 1. Load Data & Build Pipeline
We load a small sample of parsed SEC 10-K filings and build the agent pipeline.
Behind the scenes, this:
1. Chunks the documents.
2. Builds the ChromaDB vectorstore (or loads if cached).
3. Instantiates the tools with injected dependencies.
4. Compiles the LangGraph ReAct agent.


In [ ]:
# Load parsed JSON filings from disk
# Note: Ensure you have run the ingestion pipeline first!
from pathlib import Path
import os

# Dynamically resolve data directory regardless of whether the notebook 
# is running from the project root or the showcases folder
if Path("data/processed").exists():
    data_dir = Path("data/processed")
else:
    data_dir = Path("../../data/processed")

filings = []
if data_dir.exists():
    # Find .meta.json sidecars
    for meta_file in data_dir.rglob("*.meta.json"):
        md_file = meta_file.with_suffix("").with_suffix(".md")
        if md_file.exists():
            filings.append(ProcessedFiling.from_files(md_file, meta_file))
    
print(f"Loaded {len(filings)} filings from {data_dir.absolute()}")

# Initialize pipeline
print("Building Agent RAG Pipeline...")
pipeline = AgentRAGPipeline()
pipeline.build(filings)
print("Pipeline built successfully!")


## 2. Single-Step Query (Targeted Retrieval)
Let's ask a question that requires pointing to a specific document section.
Notice how the agent uses `search_section` instead of generic retrieval.


In [ ]:
query1 = "What are the key risk factors for Apple (AAPL) in FY2024?"
print(f"Query: {query1}\n")

res1 = pipeline.query(query1)

print("\n" + "="*50)
print(f"Answer:\n{res1.answer}")
print("="*50 + "\n")

print(f"Tokens Used: {res1.metrics.token_usage.total_tokens}")
print(f"Latency: {res1.metrics.latency_seconds:.2f}s")
print(f"Steps: {res1.metrics.num_steps}")
print(f"Was revised: {res1.was_revised}")
if res1.reflection_verdict is not None:
    print(f"Reflection status: {res1.reflection_verdict.status}")
    print(f"Reflection issues: {res1.reflection_verdict.issues}")

print("\nTool Call Log:")
for i, tc in enumerate(res1.tool_calls_log, 1):
    print(f"{i}. {tc['tool']} -> {tc['args']}")


## 3. Multi-Step Query (Retrieval + Math)
Here we ask a question requiring the agent to find revenue numbers for multiple companies and then calculate the difference.


In [ ]:
# Note: Edit the tickers below based on the filings you actually loaded
query2 = "What is the difference in total revenue between AAPL (FY2024) and MSFT (FY2024) in billions?"
print(f"Query: {query2}\n")

res2 = pipeline.query(query2)

print("\n" + "="*50)
print(f"Answer:\n{res2.answer}")
print("="*50 + "\n")

print(f"Tokens Used: {res2.metrics.token_usage.total_tokens}")
print(f"Latency: {res2.metrics.latency_seconds:.2f}s")
print(f"Steps: {res2.metrics.num_steps}")
print(f"Was revised: {res2.was_revised}")
if res2.reflection_verdict is not None:
    print(f"Reflection status: {res2.reflection_verdict.status}")
    print(f"Reflection issues: {res2.reflection_verdict.issues}")

print("\nTool Call Log:")
for i, tc in enumerate(res2.tool_calls_log, 1):
    print(f"{i}. {tc['tool']} -> {tc['args']}")


## 4. Ablation: Disable Reflection

For A/B comparisons between the raw ReAct baseline and the reflection-augmented System 2, the verifier can be toggled off via `config_override`. Retrieval hyperparameters stay identical (inherited from `best_config.yaml`); only the reflection step is removed.

In [ ]:
# Build a reflection-off variant for ablation. Uses the same retrieval stack
# and the same tools — only the post-answer verifier is disabled.
ablation_pipeline = AgentRAGPipeline(
    config_override={"reflection_enabled": False}
)
ablation_pipeline.build(filings)

res_ablation = ablation_pipeline.query(query1)

print(f"Reflection enabled (baseline): was_revised={res1.was_revised}, "
      f"tokens={res1.metrics.token_usage.total_tokens}")
print(f"Reflection disabled (ablation): was_revised={res_ablation.was_revised}, "
      f"tokens={res_ablation.metrics.token_usage.total_tokens}")
print(f"Token delta attributable to reflection: "
      f"{res1.metrics.token_usage.total_tokens - res_ablation.metrics.token_usage.total_tokens}")